🧠 Project: Multi-Tool AI Agent for Bangladesh
🎯 Objective
Build an AI Agent that can:
Answer data-specific queries from three Bangladesh datasets:
Institutional Information
Hospitals
Restaurants
Use a Web Search Tool for general knowledge (e.g., definitions, policies, cultural context).
📁 Provided Datasets
We’ll use these HuggingFace datasets:
Institutional Information of Bangladesh  
🔗 https://huggingface.co/datasets/Mahadih534/Institutional-Information-of-Bangladesh
All Bangladeshi Hospitals  
🔗 https://huggingface.co/datasets/Mahadih534/all-bangladeshi-hospitals
Bangladeshi Restaurant Data  
🔗 https://huggingface.co/datasets/Mahadih534/Bangladeshi-Restaurant-Data
🛠️ Project Tasks
1. Convert CSVs to SQLite DBs
For each dataset, create a SQLite database:
institutions.db → Table: institutions
hospitals.db → Table: hospitals
restaurants.db → Table: restaurants
Make sure:
Column names are meaningful (name, location, type, capacity, etc.)
Column types are set correctly (TEXT, INTEGER, REAL).
2. Build DB-specific Tools
Create LangChain tools for each DB:
InstitutionsDBTool → Queries about universities, colleges, govt institutions.
HospitalsDBTool → Queries about hospitals, beds, doctors, facilities.
RestaurantsDBTool → Queries about restaurants, cuisine, ratings, locations.
Each tool:
Connects to its DB
Executes SQL queries
Returns results in natural language.
3. Add a Web Search Tool
WebSearchTool → General knowledge  
Example: “What is the healthcare policy in Bangladesh?”
Uses SerpAPI / Tavily / Bing API  / any search tool.
4. Main AI Agent Logic
Build a Main Agent with LangChain Agent Executor:
🧮 Data/statistics queries → Route to DB tools
Example: “How many hospitals are in Dhaka?” → HospitalsDBTool
🌐 General knowledge queries → Route to Web Search tool
Example: “What is the role of DGHS in Bangladesh?” → WebSearchTool
📦 Submission Requirements
GitHub Repo with full codebase
README with instructions
Optional: Google Colab link for demo

🔍 Example Queries (Might not related to the provided db)

Query
Tool Used
Example Answer
“List top 10 hospitals in Dhaka with bed capacity.”
HospitalsDBTool
Returns hospital names + bed counts
“Which universities in Bangladesh offer medical degrees?”
InstitutionsDBTool
Returns list of institutions
“Find restaurants in Chattogram serving biryani.”
RestaurantsDBTool
Returns restaurant names + addresses
“What is the healthcare policy of Bangladesh?”
WebSearchTool
Returns web-based info
“How many government institutions are in Rajshahi?”
InstitutionsDBTool
Returns count

Answer:

Install Libraries

In [ ]:
!pip install -q langchain langchain-community langchain-openai
!pip install -q datasets pandas sqlalchemy ddgs

In [ ]:
!pip install -q ddgs

Import Librabries

In [ ]:
import pandas as pd
import sqlite3
from datasets import load_dataset

Load Hugging Face Datasets

In [ ]:
from datasets import load_dataset

institutions = load_dataset("Mahadih534/Institutional-Information-of-Bangladesh")
hospitals = load_dataset("Mahadih534/all-bangladeshi-hospitals")
restaurants = load_dataset("Mahadih534/Bangladeshi-Restaurant-Data")

Convert CSV to SQLite

In [ ]:
import pandas as pd

inst_df = pd.DataFrame(institutions["train"])
hospital_df = pd.DataFrame(hospitals["train"])
restaurant_df = pd.DataFrame(restaurants["train"])

Create Database Tools

In [ ]:
import sqlite3

inst_df.to_sql("institutions", sqlite3.connect("institutions.db"), if_exists="replace", index=False)
hospital_df.to_sql("hospitals", sqlite3.connect("hospitals.db"), if_exists="replace", index=False)
restaurant_df.to_sql("restaurants", sqlite3.connect("restaurants.db"), if_exists="replace", index=False)

12703

Database Testing

In [ ]:
conn = sqlite3.connect("hospitals.db")
pd.read_sql("SELECT * FROM hospitals LIMIT 5", conn)

,Id,Name,Name (Bangla),Code,Agency,Type,Division,District,City Corporation,Upazila,Paurasava,Union,Private
0,1,Dhaka Divisional Health Office,"পরিচালক (স্বাস্থ্য) এর কার্যালয়, ঢাকা বিভাগ, ঢাকা",10000001.0,DGHS,Divisional Level Office,Dhaka,Dhaka,Dhaka South City Corporation,Motijheel,None,None,0
1,2,Directorate General of Health Services (DGHS),"স্বাস্থ্য অধিদপ্তর, মহাখালী",10000002.0,DGHS,Directorate General or Directorate,Dhaka,Dhaka,Dhaka North City Corperation,Banani,None,None,0
2,3,"Institute of Epidemiology, Disease Control & R...","রোগতত্ব, রোগ নিয়ন্ত্রন ও গবেষণা ইনষ্টিটিউট ( আ...",10000003.0,DGHS,Public Health Institution,Dhaka,Dhaka,Dhaka North City Corperation,Banani,None,None,0
3,4,Institute of Public Health (IPH),জনস্বাস্থ্য ইনস্টিটিউট,10000004.0,DGHS,Public Health Institution,Dhaka,Dhaka,Dhaka North City Corperation,Banani,None,None,0
4,5,Institute Of Public Health Nutrition (IPHN),জনস্বাস্থ্য পুষ্টি প্রতিষ্ঠান,10000005.0,DGHS,Public Health Institution,Dhaka,Dhaka,None,None,None,None,0


SQL Helping Functional

In [ ]:
def run_query(db, query):
    conn = sqlite3.connect(db)
    result = pd.read_sql(query, conn)
    conn.close()
    return result

Tools

In [ ]:
from langchain_core.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

Institutional Toolkit

In [ ]:
def institutions_tool(query):
    q = query.lower()

    if "rajshahi" in q and "government" in q:
        sql = """
        SELECT COUNT(*) AS Total
        FROM institutions
        WHERE DISTRICT='Rajshahi'
        """

    elif "medical" in q:
        sql = """
        SELECT `INSTITUTE NAME`, DISTRICT
        FROM institutions
        WHERE `INSTITUTE_TYPE` LIKE '%Medical%'
        """

    elif "dhaka" in q:
        sql = """
        SELECT `INSTITUTE NAME`, DISTRICT
        FROM institutions
        WHERE DISTRICT='Dhaka'
        LIMIT 10
        """

    else:
        sql = """
        SELECT `INSTITUTE NAME`, DISTRICT
        FROM institutions
        LIMIT 10
        """

    return run_query("institutions.db", sql)

Hospitalize Toolkit

In [ ]:
def hospitals_tool(query):
    q = query.lower()

    if "dhaka" in q and ("top" in q or "list" in q):
        sql = """
        SELECT Name, District, Division, Type
        FROM hospitals
        WHERE District='Dhaka'
        LIMIT 10
        """

    elif "how many" in q or "count" in q:
        sql = """
        SELECT COUNT(*) AS Total_Hospitals
        FROM hospitals
        WHERE District='Dhaka'
        """

    else:
        sql = """
        SELECT Name, District, Division
        FROM hospitals
        LIMIT 10
        """

    return run_query("hospitals.db", sql)

Restaurants Toolkit

In [ ]:
def restaurants_tool(query):
    q = query.lower()

    if "top" in q or "best" in q:
        sql = """
        SELECT name,address,rating
        FROM restaurants
        ORDER BY rating DESC
        LIMIT 10
        """

    elif "popular" in q:
        sql = """
        SELECT name,address,number_of_reviews
        FROM restaurants
        ORDER BY number_of_reviews DESC
        LIMIT 10
        """

    elif "chattogram" in q:
        sql = """
        SELECT name,address,rating
        FROM restaurants
        WHERE address LIKE '%Chattogram%'
        LIMIT 10
        """

    else:
        sql = """
        SELECT name,address,rating
        FROM restaurants
        LIMIT 10
        """

    return run_query("restaurants.db", sql)

Web Search Toolkit

In [ ]:
search_tool = Tool(
    name="WebSearch",
    func=DuckDuckGoSearchRun().run,
    description="General knowledge questions"
)

Listing Tools

In [ ]:
tools = {
    "hospital": hospitals_tool,
    "institution": institutions_tool,
    "restaurant": restaurants_tool,
    "web": search_tool
}

Main Agent


In [ ]:
def agent(query):

    q = query.lower()

    if any(word in q for word in ["hospital", "bed"]):
        return hospitals_tool(query)

    elif any(word in q for word in ["university", "college", "institution", "government"]):
        return institutions_tool(query)

    elif any(word in q for word in ["restaurant", "biryani", "food"]):
        return restaurants_tool(query)

    else:
        return search_tool.run(query)

Test Example Queries

In [ ]:
agent("List top 10 hospitals in Dhaka with bed capacity")

,Name,District,Division,Type
0,Dhaka Divisional Health Office,Dhaka,Dhaka,Divisional Level Office
1,Directorate General of Health Services (DGHS),Dhaka,Dhaka,Directorate General or Directorate
2,"Institute of Epidemiology, Disease Control & R...",Dhaka,Dhaka,Public Health Institution
3,Institute of Public Health (IPH),Dhaka,Dhaka,Public Health Institution
4,Institute Of Public Health Nutrition (IPHN),Dhaka,Dhaka,Public Health Institution
5,National Institute Of Cancer Research And Hosp...,Dhaka,Dhaka,Postgraduate Institute & Hospital
6,National Institute Of Cardiovascular Disease (...,Dhaka,Dhaka,Postgraduate Institute & Hospital
7,National Institute of Diseases of the Chest & ...,Dhaka,Dhaka,Postgraduate Institute & Hospital
8,National Institute of Kidney Disease and Urolo...,Dhaka,Dhaka,Postgraduate Institute & Hospital
9,National Institute Of Mental Health (NIMH),Dhaka,Dhaka,Postgraduate Institute & Hospital


In [ ]:
agent("Show top rated restaurants")

,name,address,rating
0,Salma Varaitis Store,"Kawkhali bowlakanda, কাউখালি, Bangladesh",5.0
1,হাজী বিরিয়ানি হাউজ,"Charkhali - Mathbaria – Patharghata Rd, Mathba...",5.0
2,নিউ মুসলিম সুইটস এণ্ড বেকারি,"সদর রোড, Mathbaria, Bangladesh",5.0
3,Sharif food fair,"7XQ5+WJX, Mathbaria, Bangladesh",5.0
4,সোহেল রানা কাটোল ফাম,"College Rd, Mathbaria, Bangladesh",5.0
5,মোঃরাছেল স্টোর,"7XR5+R63, Mathbaria, Bangladesh",5.0
6,New Muslim Sweets & bekare,"7XQ5+FFQ, Mathbaria, Bangladesh",5.0
7,"পুরাতন জিন্দেগী হোটেল, মঠবাড়িয়া","376 Charkhali - Mathbaria – Patharghata Rd, Ma...",5.0
8,Ruposi Bangla hotel & Restaurant,"1 Sabujnajar Rd, Mathbaria 8560, Bangladesh",5.0
9,প্রাণকেন্দ্র (পরিমল দার গুগলী টি স্টল),"Dirgha Batiakatha Rd, Daulbari Dobra Union 854...",5.0


In [ ]:
agent("What is the healthcare policy of Bangladesh?")

"To ensure an effective health care system that responds to the need of a healthy nation, health policy provides the vision and mission for development. Pursuit of such policy will fulfill the demands of the people of the country, while health service providers will be encouraged and inspired. These policy documents are organized and listed currently under 19 categories which are the thematic areas of the International Health Regulation (IHR). The Categories are; • Legal Instruments • Financing • IHR Coordination, National IHR Focal Point Functions and Advocacy • Antimicrobial Resistance (AMR) • Zoonotic Disaese • Food safety This Compact is endorsed by the Government of Bangladesh and its development partners in recognition of shared commitment to strengthening the health system and accelerating progress toward Universal Health Coverage. Nov 8, 2024 · Explore the intricacies of Bangladesh's healthcare system, a dual structure encompassing both public and private sectors. This blog pos

In [ ]:
agent("Find popular restaurants")

,name,address,number_of_reviews
0,Hotel Noorjahan,"Holding No. 618, Near Dhaka - Chittagong Hwy, ...",17655.0
1,Star Kebab & Restaurant,"House-15 Rd no. 17, Block- C, Dhaka 1213, Bang...",16992.0
2,Star Kebab & Restaurant,"House-15 Rd no. 17, Block- C, Dhaka 1213, Bang...",16992.0
3,Star Kebab & Restaurant,"House-15 Rd no. 17, Block- C, Dhaka 1213, Bang...",16992.0
4,Star Kebab & Restaurant,"House-15 Rd no. 17, Block- C, Dhaka 1213, Bang...",16992.0
5,Sultan's Dine Dhanmondi,Green Akshay Plaza 1st Floor Satmasjid Road 14...,15622.0
6,Sultan's Dine Dhanmondi,Green Akshay Plaza 1st Floor Satmasjid Road 14...,15622.0
7,Sultan's Dine Dhanmondi,Green Akshay Plaza 1st Floor Satmasjid Road 14...,15622.0
8,Sultan's Dine Dhanmondi,Green Akshay Plaza 1st Floor Satmasjid Road 14...,15622.0
9,Sultan's Dine Dhanmondi,Green Akshay Plaza 1st Floor Satmasjid Road 14...,15622.0


Example Query

In [ ]:
print(agent("How many hospitals are in Dhaka?"))

   Total_Hospitals
0             2919


In [ ]:
print(agent("List top 10 hospitals in Dhaka"))

                                                Name District Division  \
0                     Dhaka Divisional Health Office    Dhaka    Dhaka   
1      Directorate General of Health Services (DGHS)    Dhaka    Dhaka   
2  Institute of Epidemiology, Disease Control & R...    Dhaka    Dhaka   
3                   Institute of Public Health (IPH)    Dhaka    Dhaka   
4        Institute Of Public Health Nutrition (IPHN)    Dhaka    Dhaka   
5  National Institute Of Cancer Research And Hosp...    Dhaka    Dhaka   
6  National Institute Of Cardiovascular Disease (...    Dhaka    Dhaka   
7  National Institute of Diseases of the Chest & ...    Dhaka    Dhaka   
8  National Institute of Kidney Disease and Urolo...    Dhaka    Dhaka   
9         National Institute Of Mental Health (NIMH)    Dhaka    Dhaka   

                                 Type  
0             Divisional Level Office  
1  Directorate General or Directorate  
2           Public Health Institution  
3           Public Health

In [ ]:
print(agent("How many government institutions are in Rajshahi?"))

   Total
0      0


In [ ]:
print(agent("Show top rated restaurants"))

                                     name  \
0                    Salma Varaitis Store   
1                      হাজী বিরিয়ানি হাউজ   
2            নিউ মুসলিম সুইটস এণ্ড বেকারি   
3                        Sharif food fair   
4                    সোহেল রানা কাটোল ফাম   
5                          মোঃরাছেল স্টোর   
6              New Muslim Sweets & bekare   
7         পুরাতন জিন্দেগী হোটেল, মঠবাড়িয়া   
8        Ruposi Bangla hotel & Restaurant   
9  প্রাণকেন্দ্র (পরিমল দার গুগলী টি স্টল)   

                                             address  rating  
0           Kawkhali bowlakanda, কাউখালি, Bangladesh     5.0  
1  Charkhali - Mathbaria – Patharghata Rd, Mathba...     5.0  
2                     সদর রোড, Mathbaria, Bangladesh     5.0  
3                    7XQ5+WJX, Mathbaria, Bangladesh     5.0  
4                  College Rd, Mathbaria, Bangladesh     5.0  
5                    7XR5+R63, Mathbaria, Bangladesh     5.0  
6                    7XQ5+FFQ, Mathbaria, Bangladesh     5.0  


In [ ]:
print(agent("What is the healthcare policy of Bangladesh?"))

March 31, 2026 - Bangladesh falls into the "bad" category when evaluating the right to reproductive health because the nation is fulfilling only 78.8% of what the nation is expected to achieve based on the resources (income) it has available. To ensure equitable healthcare for every resident in Bangladesh, ... 2 weeks ago - The Ministry of Health and Family Welfare (Bengali: স্বাস্থ্য ও পরিবার কল্যাণ মন্ত্রণালয়; Sbāsthya ō paribār kalyāṇ Montronaloya) is a Bangladesh government ministry charged with health policy in Bangladesh. December 6, 2025 - the quality and availability of essential services, invest in a skilled and team-based workforce, and · promote financial protection mechanisms, and leverage private sector engagement and innovation. Development partners, civil society, communities, and the private sector are encouraged to align their · support with this national direction as Bangladesh advances toward UHC. December 12, 2025 - UHC is not only a health goal, but also a corners

Readme.md

In [ ]:
%%writefile README.md
# Module 22 Assignment

# Bangladesh Multi-Tool AI Agent

# ASHIQ RAHMAN CHOWDHURY

## Objective

This project builds a Multi-Tool AI Agent that answers questions from three Bangladesh datasets and uses web search for general knowledge.

## Datasets

- Institutional Information of Bangladesh
- All Bangladeshi Hospitals
- Bangladeshi Restaurant Data

## Features

- Query Institutional Information
- Query Hospital Information
- Query Restaurant Information
- Web Search using DuckDuckGo

## Technologies

- Python
- SQLite
- Pandas
- LangChain
- DuckDuckGo Search

## How to Run

1. Install the required libraries.
2. Run all notebook cells.
3. Test using the example queries.

## Example Queries

- How many hospitals are in Dhaka?
- List top 10 hospitals in Dhaka.
- How many government institutions are in Rajshahi?
- Show top rated restaurants.
- What is the healthcare policy of Bangladesh?

Overwriting README.md


requirements.txt

In [ ]:
%%writefile requirements.txt
pandas
datasets
langchain
langchain-community
langchain-openai
duckduckgo-search
ddgs
sqlalchemy

Writing requirements.txt


In [ ]:
import os

print(os.listdir())

['.config', 'restaurants.db', 'hospitals.db', 'requirements.txt', 'README.md', 'institutions.db', 'sample_data']
